In [1]:
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.1 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
import openai
import pandas as pd
import re
import os
from sklearn.metrics import precision_recall_fscore_support, classification_report, accuracy_score
import requests

In [3]:
user_secrets = UserSecretsClient()
OPEN_AI_POS_1 = user_secrets.get_secret("OPEN_AI_POS_1")
GEMINI_API_KEY  = user_secrets.get_secret("genmini_api_key")

openai.api_key = OPEN_AI_POS_1

In [ ]:
# Use this for 1L sentences
folder_path = '/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences'
files = os.listdir(folder_path)
txt_files = [file for file in files if file.endswith('.txt')]
len(txt_files)

In [ ]:
taggedData = []
for file in txt_files:
    with open("/kaggle/input/konkani-ilci-pos-tagged-1lakh-sentences/" + file, 'r', encoding='utf-8') as file:
        content = file.read()
    content = content.split("\n")
    content = content[1:]  # Skip the first line
    taggedDataFile = [line.split('\t')[1] for line in content if 'Value' not in line.split('\t')[1]]
    taggedData.append(taggedDataFile)
    
allTaggedData = [item for sublist in taggedData for item in sublist]
allTaggedData[:2]

In [4]:
df_ref = pd.read_excel('/kaggle/input/konkani-pos-tagged-reference-sentences/anwani_ref_sentences.xlsx')
df_ref.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,Unnamed: 3
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,NaN
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,NaN
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,NaN
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,NaN
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,NaN


In [5]:
df_ref = df_ref.iloc[:, :3]
df_ref.head()

,sentences_cleaned,predicted_sentences,tagged_sentences
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...


In [6]:
df_ref = df_ref.iloc[:100, :3]
df_ref.tail()

,sentences_cleaned,predicted_sentences,tagged_sentences
95,बस . लागीं लागीं कोण आसा अशें दिसना .,बस\N_NN .\RD_PUNC लागीं\N_NST लागीं\N_NST कोण\...,बस\V_VM_VF .\RD_PUNC लागीं\N_NST लागीं\N_NST क...
96,सायबान आमकां आनीक थोडे दीस कामार दवरल्यार बरें...,सायबान\N_NN आमकां\PR_PRP आनीक\QT_QTF थोडे\QT_Q...,सायबान\N_NN आमकां\PR_PRP आनीक\RP_INTF थोडे\QT_...
97,काम करना आसतना किद्याक पोंसता आशिल्लो तो ?,काम\N_NN करना\N_NNP आसतना\V_VM_VNF किद्याक\PR_...,काम\N_NN करना\V_VM_VNF आसतना\V_VM_VNF किद्याक\...
98,तांचें बरें न्ही ? तांकां खंयच चलत वच्चें पडले...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...
99,तांचें किदें बाये ? आमचे सारकी काटकसरीची जीण न...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...


In [6]:
ref_sentences = df_ref['ref_sentences'].tolist()
print(len(ref_sentences))
allTaggedData = ref_sentences[:100]
print(len(allTaggedData))

101
100


In [7]:
allTaggedData[:2]

['मनशाच्या\\N_NN शरिरांत\\N_NN 206\\QT_QTC हाडां\\N_NN आसात\\V_VM_VF .\\RD_PUNC',
 'मनशाचें\\N_NN काळीज\\N_NN दिसाक\\N_NN सुमार\\RB 1\\QT_QTC लाख\\N_NN फावटीं\\N_NN ठोके\\N_NN दिता\\V_VM_VF .\\RD_PUNC']

In [8]:
df = pd.DataFrame(allTaggedData, columns=['actual_tags'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes


In [9]:
def preprocess_sentence(sentence):
    words, tags = [], []
    for token in sentence.split():
        if "\\" in token:
            word, tag = token.rsplit("\\", 1)  # Split at last occurrence of '\'
            words.append(word)
            tags.append(tag)
    return words, tags

# Apply preprocessing to extract words & POS tags
df["sentence"], df["labels"] = zip(*df["actual_tags"].map(preprocess_sentence))

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   actual_tags  100 non-null    object
 1   sentence     100 non-null    object
 2   labels       100 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB


In [12]:
df = df.drop(columns=['labels'])
df.head()

,actual_tags,sentence
0,मनशाच्या\N_NN शरिरांत\N_NN 206\QT_QTC हाडां\N_...,"[मनशाच्या, शरिरांत, 206, हाडां, आसात, .]"
1,मनशाचें\N_NN काळीज\N_NN दिसाक\N_NN सुमार\RB 1\...,"[मनशाचें, काळीज, दिसाक, सुमार, 1, लाख, फावटीं,..."
2,सूर्य\N_NNP पृथ्वी\N_NNP परस\PSP 3\QT_QTC लाख\...,"[सूर्य, पृथ्वी, परस, 3, लाख, परस, चड, पटीन, आक..."
3,नील\N_NNP आर्मस्ट्राँग\N_NNP हो\PR_PRP 1969\N_...,"[नील, आर्मस्ट्राँग, हो, 1969, वर्सा, चंद्राचेर..."
4,पृथ्वीचो\N_NNP सगल्यांत\RP_INTF व्हडलो\JJ आनी\...,"[पृथ्वीचो, सगल्यांत, व्हडलो, आनी, एकमेव, सैमीक..."


In [7]:
df = df_ref

In [8]:
df.head()

,sentences_cleaned,predicted_sentences,tagged_sentences
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...


In [9]:
def get_pos_tags_from_gpt(sentence):
    # BIS Tagset for Konkani (added knowledge base)
    bis_tagset = """
    1. Noun (N)
    Noun Common (NN): N_NN
    Noun Proper (NNP): N_NNP
    Noun Locative (NST): N_NST
    
    2. Pronoun (PR)
    Pronoun Personal (PRP): PR_PRP
    Pronoun Reflexive (PRF): PR_PRF
    Pronoun Relative (PRL): PR_PRL
    Pronoun Reciprocal (PRC): PR_PRC
    Pronoun Wh-word (PRQ): PR_PRQ
    Pronoun Indefinite (PRI): PR_PRI
    
    3. Demonstrative (DM)
    Demonstrative Deictic (DMD): DM_DMD
    Demonstrative Relative (DMR): DM_DMR
    Demonstrative Wh-word (DMQ): DM_DMQ
    Demonstrative Indefinite (DMI): DM_DMI
    
    4. Verb (V)
    Main Verb (VM): V_VM
    Finite Main Verb (VF): V_VM_VF
    Non-finite Main Verb (VNF): V_VM_VNF
    Infinitive Verb (VINF): V_VM_VINF
    Gerund Verb (VNG): V_VM_VNG
    Auxiliary Verb (VAUX):
    Finite Auxiliary Verb (VAUX_F): V_VAUX_VF
    Non-finite Auxiliary Verb (VAUX_NF): V_VAUX_VNF
    
    5. Adjective (JJ)
    Adjective (JJ): JJ
    
    6. Adverb (RB)
    Adverb (RB): RB
    
    7. Postposition (PSP)
    Postposition (PSP): PSP
    
    8. Conjunction (CC)
    Conjunction Coordinator (CCD): CC_CCD
    Conjunction Subordinator (CCS): CC_CCS
    
    9. Particles (RP)
    Default Particle (RPD): RP_RPD
    Classifier (CL): RP_CL
    Interjection (INJ): RP_INJ
    Intensifier (INTF): RP_INTF
    Negation (NEG): RP__NEG
    
    10. Quantifiers (QT)
    General Quantifier (QTF): QT_QTF
    Cardinal Quantifier (QTC): QT_QTC
    Ordinal Quantifier (QTO): QT_QTO
    
    11. Residuals (RD)
    Foreign Word (RDF): RD_RDF
    Symbol (SYM): RD_SYM
    Punctuation (PUNC): RD_PUNC
    Unknown (UNK): RD_UNK
    Echowords (ECH): RD_ECH
    """

    prompt = f"""
    You are a expert POS tagger that provides POS tagging for Konkani sentences. Below is the BIS Tagset for Konkani:
    {bis_tagset}
    
    Tag each word in the sentence with its POS tag using the below BIS tagset for Konkani:
    [ "N_NN", "N_NNP", "N_NST", "PR_PRP", "PR_PRF", "PR_PRL", "PR_PRC", "PR_PRQ", "PR_PRI", 
      "DM_DMD", "DM_DMR", "DM_DMQ", "DM_DMI", "V_VM", "V_VM_VF", "V_VM_VNF", "V_VM_VINF", "V_VM_VNG", 
      "V_VAUX_VF", "V_VAUX_VNF", "JJ", "RB", "PSP", "CC_CCD", "CC_CCS", "RP_RPD", "RP_CL", "RP_INJ", 
      "RP_INTF", "RP_NEG", "QT_QTF", "QT_QTC", "QT_QTO", "RD_RDF", "RD_SYM", "RD_PUNC", "RD_UNK", 
      "RD_ECH" ]
    
    Format: word\\POS\n\nSentence: {sentence}
    Note: Output should only consist of POS Tagged Sentence. 
    """

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert POS tagger."},
            {"role": "user", "content": prompt}
        ]
    )

    gpt_output = response["choices"][0]["message"]["content"]
    # Get the response and return the content
    return gpt_output

In [85]:
def get_gemini_pos_tags(sentence):
    bis_tagset = """
    1. Noun (N)
    Noun Common (NN): N_NN
    Noun Proper (NNP): N_NNP
    Noun Locative (NST): N_NST
    
    2. Pronoun (PR)
    Pronoun Personal (PRP): PR_PRP
    Pronoun Reflexive (PRF): PR_PRF
    Pronoun Relative (PRL): PR_PRL
    Pronoun Reciprocal (PRC): PR_PRC
    Pronoun Wh-word (PRQ): PR_PRQ
    Pronoun Indefinite (PRI): PR_PRI
    
    3. Demonstrative (DM)
    Demonstrative Deictic (DMD): DM_DMD
    Demonstrative Relative (DMR): DM_DMR
    Demonstrative Wh-word (DMQ): DM_DMQ
    Demonstrative Indefinite (DMI): DM_DMI
    
    4. Verb (V)
    Main Verb (VM): V_VM
    Finite Main Verb (VF): V_VM_VF
    Non-finite Main Verb (VNF): V_VM_VNF
    Infinitive Verb (VINF): V_VM_VINF
    Gerund Verb (VNG): V_VM_VNG
    Auxiliary Verb (VAUX):
    Finite Auxiliary Verb (VAUX_F): V_VAUX_VF
    Non-finite Auxiliary Verb (VAUX_NF): V_VAUX_VNF
    
    5. Adjective (JJ)
    Adjective (JJ): JJ
    
    6. Adverb (RB)
    Adverb (RB): RB
    
    7. Postposition (PSP)
    Postposition (PSP): PSP
    
    8. Conjunction (CC)
    Conjunction Coordinator (CCD): CC_CCD
    Conjunction Subordinator (CCS): CC_CCS
    
    9. Particles (RP)
    Default Particle (RPD): RP_RPD
    Classifier (CL): RP_CL
    Interjection (INJ): RP_INJ
    Intensifier (INTF): RP_INTF
    Negation (NEG): RP__NEG
    
    10. Quantifiers (QT)
    General Quantifier (QTF): QT_QTF
    Cardinal Quantifier (QTC): QT_QTC
    Ordinal Quantifier (QTO): QT_QTO
    
    11. Residuals (RD)
    Foreign Word (RDF): RD_RDF
    Symbol (SYM): RD_SYM
    Punctuation (PUNC): RD_PUNC
    Unknown (UNK): RD_UNK
    Echowords (ECH): RD_ECH
    """

    prompt = f"""
    You are a expert POS tagger that provides POS tagging for Konkani sentences. Below is the BIS Tagset for Konkani:
    {bis_tagset}
    
    Tag each word in the sentence with its POS tag using the below BIS tagset for Konkani:
    [ "N_NN", "N_NNP", "N_NST", "PR_PRP", "PR_PRF", "PR_PRL", "PR_PRC", "PR_PRQ", "PR_PRI", 
      "DM_DMD", "DM_DMR", "DM_DMQ", "DM_DMI", "V_VM", "V_VM_VF", "V_VM_VNF", "V_VM_VINF", "V_VM_VNG", 
      "V_VAUX_VF", "V_VAUX_VNF", "JJ", "RB", "PSP", "CC_CCD", "CC_CCS", "RP_RPD", "RP_CL", "RP_INJ", 
      "RP_INTF", "RP_NEG", "QT_QTF", "QT_QTC", "QT_QTO", "RD_RDF", "RD_SYM", "RD_PUNC", "RD_UNK", 
      "RD_ECH" ]
    
    Format: word\\POS\n\nSentence(There should be a "\" between word and the POS strictly and not _): {sentence}
    Note: Output should only consist of POS Tagged Sentence. 
    """
    url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent"
    headers = {"Content-Type": "application/json"}
    params = {"key": GEMINI_API_KEY}
    payload = {
        "contents": [{"parts": [{"text": prompt}]}]
    }

    response = requests.post(url, headers=headers, params=params, json=payload)

    if response.status_code == 200:
        result = response.json()
        return result.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")
    else:
        return f"Error: {response.status_code} - {response.text}"

In [86]:
#sentence = "ताची आनीकूय कांय सिनेमां प्रस्तावीत आसा ताचेर अजून काम सुरू जावंक ना ."
sentence = "ह्या अभिनेत्या भितर स्वताचें हास्य प्रतिभे शिवाय आनीक एक सैमीक कला आशिल्ली , जाका लागून ताच्या जैतांत भर पडली ."
gemini_output = get_gemini_pos_tags(sentence)
print(gemini_output)

ह्या\DM_DMD अभिनेत्या\N_NN भितर\PSP स्वताचें\PR_PRF हास्य\N_NN प्रतिभे\N_NN शिवाय\PSP आनीक\QT_QTF एक\QT_QTC सैमीक\JJ कला\N_NN आशिल्ली\V_VM_VF,\RD_PUNC जाका\DM_DMD लागून\PSP ताच्या\PR_PRP जैतांत\N_NN भर\N_NN पडली\V_VM_VF.\RD_PUNC



In [126]:
df_sample = df[90:100]
df_sample

,sentences_cleaned,predicted_sentences,tagged_sentences
90,बरें आसा . चल तूं तुज्या कामाक चल . हांगा थंय ...,बरें\RB आसा\V_VM_VF .\RD_PUNC चल\V_VM_VF तूं\P...,बरें\RB आसा\V_VM_VF .\RD_PUNC चल\V_VM_VF तूं\P...
91,"हे वटेन कोण ना दिसता रे , चल ते वटेन पळोवया . ...",हे\DM_DMD वटेन\N_NN कोण\PR_PRQ ना\RP_NEG दिसता...,हे\DM_DMD वटेन\N_NN कोण\PR_PRQ ना\RP_NEG दिसता...
92,"हय , चल या . खंयचे चोर वाटेन वतात देव जाण मेल्...","हय\V_VM_VF ,\RD_PUNC चल\V_VM_VF या\N_NN .\RD_P...","हय\V_VM_VF ,\RD_PUNC चल\V_VM_VNF या\V_VAUX_VF ..."
93,घडये शीम लागींच आसतली . परत इल्लो सुशेग घेवन भ...,घडये\RB शीम\N_NN लागींच\N_NST आसतली\V_VM_VF .\...,घडये\RB शीम\N_NN लागींच\N_NST आसतली\V_VM_VF .\...
94,"हय , म्हज्या पोटांतय मातशें दुखूंक लागला . बसत...","हय\V_VM_VF ,\RD_PUNC म्हज्या\PR_PRP पोटांतय\N_...","हय\V_VM_VF ,\RD_PUNC म्हज्या\PR_PRP पोटांतय\N_..."
95,बस . लागीं लागीं कोण आसा अशें दिसना .,बस\N_NN .\RD_PUNC लागीं\N_NST लागीं\N_NST कोण\...,बस\V_VM_VF .\RD_PUNC लागीं\N_NST लागीं\N_NST क...
96,सायबान आमकां आनीक थोडे दीस कामार दवरल्यार बरें...,सायबान\N_NN आमकां\PR_PRP आनीक\QT_QTF थोडे\QT_Q...,सायबान\N_NN आमकां\PR_PRP आनीक\RP_INTF थोडे\QT_...
97,काम करना आसतना किद्याक पोंसता आशिल्लो तो ?,काम\N_NN करना\N_NNP आसतना\V_VM_VNF किद्याक\PR_...,काम\N_NN करना\V_VM_VNF आसतना\V_VM_VNF किद्याक\...
98,तांचें बरें न्ही ? तांकां खंयच चलत वच्चें पडले...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...
99,तांचें किदें बाये ? आमचे सारकी काटकसरीची जीण न...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...


In [129]:
import time

counter = 0

def get_pos_tags_with_delay(sentence):
    global counter
    result = get_pos_tags_from_gpt(sentence)  # Call the GPT function
    counter += 1
    if counter % 2 == 0:  # Pause every 2 API calls
        time.sleep(30)
    return result


df_sample["gpt_output"] = df_sample["sentences_cleaned"].apply(get_gemini_pos_tags)
df_sample

<ipython-input-129-d6287ee82929>:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample["gpt_output"] = df_sample["sentences_cleaned"].apply(get_gemini_pos_tags)


,sentences_cleaned,predicted_sentences,tagged_sentences,gpt_output
90,बरें आसा . चल तूं तुज्या कामाक चल . हांगा थंय ...,बरें\RB आसा\V_VM_VF .\RD_PUNC चल\V_VM_VF तूं\P...,बरें\RB आसा\V_VM_VF .\RD_PUNC चल\V_VM_VF तूं\P...,बरें\JJ आसा\V_VM_VF .\RD_PUNC चल\V_VM तूं\PR_P...
91,"हे वटेन कोण ना दिसता रे , चल ते वटेन पळोवया . ...",हे\DM_DMD वटेन\N_NN कोण\PR_PRQ ना\RP_NEG दिसता...,हे\DM_DMD वटेन\N_NN कोण\PR_PRQ ना\RP_NEG दिसता...,हे\DM_DMD वटेन\N_NN कोण\PR_PRQ ना\RP__NEG दिसत...
92,"हय , चल या . खंयचे चोर वाटेन वतात देव जाण मेल्...","हय\V_VM_VF ,\RD_PUNC चल\V_VM_VF या\N_NN .\RD_P...","हय\V_VM_VF ,\RD_PUNC चल\V_VM_VNF या\V_VAUX_VF ...","हय\RP_INJ ,RD_PUNC चल\V_VM_VF या\RD_PUNC .RD_P..."
93,घडये शीम लागींच आसतली . परत इल्लो सुशेग घेवन भ...,घडये\RB शीम\N_NN लागींच\N_NST आसतली\V_VM_VF .\...,घडये\RB शीम\N_NN लागींच\N_NST आसतली\V_VM_VF .\...,घडये\RB शीम\N_NN लागींच\RB आसतली\V_VAUX_VF .\R...
94,"हय , म्हज्या पोटांतय मातशें दुखूंक लागला . बसत...","हय\V_VM_VF ,\RD_PUNC म्हज्या\PR_PRP पोटांतय\N_...","हय\V_VM_VF ,\RD_PUNC म्हज्या\PR_PRP पोटांतय\N_...","हय\RP_INJ ,\,RD_PUNC म्हज्या\PR_PRP पोटांतय\N_..."
95,बस . लागीं लागीं कोण आसा अशें दिसना .,बस\N_NN .\RD_PUNC लागीं\N_NST लागीं\N_NST कोण\...,बस\V_VM_VF .\RD_PUNC लागीं\N_NST लागीं\N_NST क...,बस\RD_PUNC . \RD_PUNC लागीं\PSP लागीं\RB कोण\P...
96,सायबान आमकां आनीक थोडे दीस कामार दवरल्यार बरें...,सायबान\N_NN आमकां\PR_PRP आनीक\QT_QTF थोडे\QT_Q...,सायबान\N_NN आमकां\PR_PRP आनीक\RP_INTF थोडे\QT_...,सायबान\N_NN आमकां\PR_PRP आनीक\RB थोडे\QT_QTF द...
97,काम करना आसतना किद्याक पोंसता आशिल्लो तो ?,काम\N_NN करना\N_NNP आसतना\V_VM_VNF किद्याक\PR_...,काम\N_NN करना\V_VM_VNF आसतना\V_VM_VNF किद्याक\...,काम\V_VM_VNG करना\V_VM_VNG आसतना\V_VM_VNG किद्...
98,तांचें बरें न्ही ? तांकां खंयच चलत वच्चें पडले...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\JJ न्ही\RP_RPD ?\RD_PUNC ता...
99,तांचें किदें बाये ? आमचे सारकी काटकसरीची जीण न...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...,तांचें\PR_PRP किदें\PR_PRQ बाये\N_NN ?\RD_PUNC...,तांचें\PR_PRP किदें\PR_PRQ बाये\RP_INJ ?\RD_PU...


In [130]:
df_total = append_new_data(df_total, df_sample)
df_total

,sentences_cleaned,predicted_sentences,tagged_sentences,gpt_output
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\PSP ना\RP__NEG ...
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP__NEG जाल्यार\CC_...
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\RB आसा\V_VM_VF म्हणून\CC_CCS ...
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCD हांगा\RB रावनय\V_VM_VNF जावचें\V_VM...
...,...,...,...,...
95,बस . लागीं लागीं कोण आसा अशें दिसना .,बस\N_NN .\RD_PUNC लागीं\N_NST लागीं\N_NST कोण\...,बस\V_VM_VF .\RD_PUNC लागीं\N_NST लागीं\N_NST क...,बस\RD_PUNC . \RD_PUNC लागीं\PSP लागीं\RB कोण\P...
96,सायबान आमकां आनीक थोडे दीस कामार दवरल्यार बरें...,सायबान\N_NN आमकां\PR_PRP आनीक\QT_QTF थोडे\QT_Q...,सायबान\N_NN आमकां\PR_PRP आनीक\RP_INTF थोडे\QT_...,सायबान\N_NN आमकां\PR_PRP आनीक\RB थोडे\QT_QTF द...
97,काम करना आसतना किद्याक पोंसता आशिल्लो तो ?,काम\N_NN करना\N_NNP आसतना\V_VM_VNF किद्याक\PR_...,काम\N_NN करना\V_VM_VNF आसतना\V_VM_VNF किद्याक\...,काम\V_VM_VNG करना\V_VM_VNG आसतना\V_VM_VNG किद्...
98,तांचें बरें न्ही ? तांकां खंयच चलत वच्चें पडले...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\RB न्ही\RP_NEG ?\RD_PUNC ता...,तांचें\PR_PRP बरें\JJ न्ही\RP_RPD ?\RD_PUNC ता...


In [131]:
#df_total = df_total.iloc[:30]
#df_total

In [137]:
df_total.to_excel("df_total_gemini.xlsx", index=False)

In [90]:
def append_new_data(df_main, df_new):
    """
    Append new rows from df_new to df_main and return the updated df_main.
    """
    df_main = pd.concat([df_main, df_new], ignore_index=True)  # Append new rows
    return df_main

In [89]:
df_total = pd.DataFrame(columns=["sentences_cleaned", "predicted_sentences","tagged_sentences", "gpt_output"])

In [132]:
import pandas as pd

def add_length_columns(df):
    # Compute lengths of actual_tags and predicted_sentences
    df['ref_sentences_length'] = df['tagged_sentences'].apply(lambda x: len(str(x).split()))
    df['joined_sent_length'] = df['gpt_output'].apply(lambda x: len(str(x).split()))
    
    # Find row indices where lengths do not match
    mismatched_rows = df[df['ref_sentences_length'] != df['joined_sent_length']].index.tolist()
    
    # Remove mismatched rows from the DataFrame
    #df = df.drop(mismatched_rows)
    
    return df, mismatched_rows

# Example usage:
df_total, mismatched_indices = add_length_columns(df_total)
mismatched_indices

[6, 18, 48, 49, 56, 59, 61, 64, 70, 82, 86, 95]

In [133]:
for ind in mismatched_indices:
    df_total = df_total.drop(ind)

df_total.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   sentences_cleaned     88 non-null     object
 1   predicted_sentences   88 non-null     object
 2   tagged_sentences      88 non-null     object
 3   gpt_output            88 non-null     object
 4   ref_sentences_length  88 non-null     int64 
 5   joined_sent_length    88 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 4.8+ KB


In [134]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

def evaluate_tagging(ref_sent, predicted_sent):
    #print("ref_sent: ", ref_sent)
    #print("predicted_sent: ", predicted_sent)
    # Handle missing values
    if not isinstance(ref_sent, str) or not isinstance(predicted_sent, str):
        return 0.0, 0.0, 0.0, 0.0  # Return default scores for missing values

    # Split sentences into word-tag pairs
    ref_tags = ref_sent.split()
    pred_tags = predicted_sent.split()

    if len(ref_tags) != len(pred_tags):
        return 0.0, 0.0, 0.0, 0.0
    

    # Extract only the tags from word\tag pairs
    ref_labels = [token.split("\\")[-1] for token in ref_tags]
    pred_labels = [token.split("\\")[-1] for token in pred_tags]

    #print(len(ref_labels), len(pred_labels))

    # Compute accuracy
    accuracy = accuracy_score(ref_labels, pred_labels)

    # Compute precision, recall, and F1-score (weighted for class imbalance)
    precision, recall, f1, _ = precision_recall_fscore_support(ref_labels, pred_labels, average='weighted', zero_division=0)

    return accuracy, precision, recall, f1, ref_labels, pred_labels

In [135]:
# Apply function to each row
df_total[["Accuracy", 
        "Precision", 
        "Recall", 
        "F1 Score", 
        "Ref Labels", 
        "Pred Labels"]] = df_total.apply(lambda row: evaluate_tagging(row["tagged_sentences"], row["gpt_output"]), 
                                       axis=1, 
                                       result_type="expand")
df_total.head()

,sentences_cleaned,predicted_sentences,tagged_sentences,gpt_output,ref_sentences_length,joined_sent_length,Accuracy,Precision,Recall,F1 Score,Ref Labels,Pred Labels
0,आयकलां ? लागीं ना आमचो गांव . चलून चलून कितलीं...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\N_NST ना\RP_NEG...,आयकलां\V_VM_VF ?\RD_PUNC लागीं\PSP ना\RP__NEG ...,14,14,0.642857,0.678571,0.642857,0.647619,"[V_VM_VF, RD_PUNC, N_NST, RP_NEG, DM_DMD, N_NN...","[V_VM_VF, RD_PUNC, PSP, RP__NEG, JJ, N_NN, RD_..."
1,दोन तीन दिसांनी जायत सुरू येरादारी . इतली कित्...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VAUX...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,दोन\QT_QTC तीन\QT_QTC दिसांनी\N_NN जायत\V_VM_V...,11,11,0.545455,0.545455,0.545455,0.545455,"[QT_QTC, QT_QTC, N_NN, V_VM_VNF, RB, N_NN, RD_...","[QT_QTC, QT_QTC, N_NN, V_VM_VF, JJ, N_NN, .RD_..."
2,आनी जाली ना जाल्यार ? कोण घेतलो आमकां घरांत ? ...,आनी\CC_CCD जाली\V_VM_VF ना\RP_NEG जाल्यार\CC_C...,आनी\CC_CCD जाली\V_VM_VNF ना\RP_NEG जाल्यार\CC_...,आनी\CC_CCD जाली\V_VM_VF ना\RP__NEG जाल्यार\CC_...,22,22,0.636364,0.628788,0.636364,0.622511,"[CC_CCD, V_VM_VNF, RP_NEG, CC_CCS, RD_PUNC, PR...","[CC_CCD, V_VM_VF, RP__NEG, CC_CCS, RD_PUNC, PR..."
3,"हांव अशें आसा म्हणून तर सांगतां , घाई नाका . आ...",हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\DM_DMD आसा\V_VM_VF म्हणून\CC_...,हांव\PR_PRP अशें\RB आसा\V_VM_VF म्हणून\CC_CCS ...,17,17,0.588235,0.720588,0.588235,0.621849,"[PR_PRP, DM_DMD, V_VM_VF, CC_CCS, CC_CCS, V_VM...","[PR_PRP, RB, V_VM_VF, CC_CCS, RP_RPD, V_VM_VF,..."
4,पूण हांगा रावनय जावचें ना . चलत रावल्यार मेळत ...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCS हांगा\N_NST रावनय\V_VM_VNF जावचें\V...,पूण\CC_CCD हांगा\RB रावनय\V_VM_VNF जावचें\V_VM...,15,15,0.666667,0.766667,0.666667,0.697778,"[CC_CCS, N_NST, V_VM_VNF, V_VM_VNF, RP_NEG, RD...","[CC_CCD, RB, V_VM_VNF, V_VM_VNF, RP_NEG, RD_PU..."


In [136]:
# Flatten all reference and predicted labels for classification report
all_ref_labels = [label for labels in df_total["Ref Labels"] for label in labels]
all_pred_labels = [label for labels in df_total["Pred Labels"] for label in labels]

# Generate classification report
print("\nClassification Report:\n")
print(classification_report(all_ref_labels, all_pred_labels, zero_division=0))


Classification Report:

              precision    recall  f1-score   support

    !RD_PUNC       0.00      0.00      0.00         0
    ,RD_PUNC       0.00      0.00      0.00         0
           .       0.00      0.00      0.00         0
    .RD_PUNC       0.00      0.00      0.00         0
    ?RD_PUNC       0.00      0.00      0.00         0
      CC_CCD       0.52      1.00      0.68        13
      CC_CCS       0.78      0.64      0.70        39
      DM_DMD       0.82      0.63      0.71        49
      DM_DMI       0.00      0.00      0.00         7
      DM_DMQ       0.67      0.13      0.22        15
          JJ       0.18      1.00      0.31        11
         NST       0.00      0.00      0.00         1
        N_NN       0.96      0.81      0.88       187
       N_NNP       0.00      0.00      0.00         1
       N_NST       0.08      0.01      0.03        68
         PRI       0.00      0.00      0.00         0
      PR_PRF       1.00      1.00      1.00         1
  